# Stage 6-2. CNN 학습 — Experiment, CuPy fallback, MLP 비교

이 노트북은 `src/models/cnn.py`와 `src/core/experiment.py`를 실습한다.

| 클래스 | 역할 |
|---|---|
| `CNN(task, seed)` | CuPy 기반 CNN — MLP와 동일한 외부 인터페이스 |
| `Experiment(config)` | `config["model"]="cnn"` 으로 CNN 파이프라인 조립 |
| `Experiment.run()` | epoch별 train/test log 반환 |
| `Predictor.predict(images)` | CNN logit → task별 예측 후처리 |

**학습 목표**
1. `config["model"]="cnn"` 설정만으로 CNN 파이프라인이 조립되는 것을 확인한다.
2. CuPy 미설치 환경에서 NumPy fallback으로 동일 코드가 실행됨을 확인한다.
3. multiclass / binary / regression 3종 task에서 CNN을 학습하고 결과를 비교한다.
4. 학습 곡선과 예측 grid를 시각화한다.
5. 같은 task에서 MLP와 CNN의 학습 결과를 비교한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt

try:
    import cupy
    _BACKEND = f"CuPy {cupy.__version__}"
except ImportError:
    _BACKEND = "NumPy (CuPy 미설치 — fallback)"

print(f"백엔드: {_BACKEND}")

from src.core.experiment import Experiment
from src.models.cnn import CNN
from src.models.mlp import MLP
from src.data.mnist import MnistDataset
from src.task import get_task_spec

DATASET_DIR = "/mnt/d/datasets/mnist"

## 1. Experiment — config["model"]="cnn" 설정

In [ ]:
config = {
    "dataset_dir": DATASET_DIR,
    "task":        "multiclass",
    "model":       "cnn",          # ← CNN 선택
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  5,
    "lr":          0.01,
}

exp = Experiment(config)

print("Experiment 구성 요소:")
print(f"  train_loader : {len(exp.train_loader)} 배치")
print(f"  test_loader  : {len(exp.test_loader)} 배치")
print(f"  model        : {type(exp.model).__name__}")
print(f"  optimizer    : {type(exp.trainer.optimizer).__name__}")
print(f"  trainer      : {type(exp.trainer).__name__}")
print(f"  evaluator    : {type(exp.evaluator).__name__}")
print(f"  predictor    : {type(exp.predictor).__name__}")
print()
print(f"  CNN 총 파라미터: {sum(p.size for p in exp.model.params):,}")
print()
print("Experiment(config)는 config['model'] 키만으로 CNN/MLP를 선택한다.")
print("Trainer, Evaluator, Predictor는 수정 없이 재사용된다.")

## 2. CNN 학습 — multiclass

In [ ]:
print("CNN multiclass 학습 시작 (5 epochs)...")
logs_mc = exp.run()

print()
print(f"{'epoch':>5} | {'train loss':>10} {'train acc':>10} | {'test loss':>10} {'test acc':>10}")
print("-" * 55)
for log in logs_mc:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>10.4f} {log['train']['metric']:>10.4f} | "
          f"{log['test']['loss']:>10.4f} {log['test']['metric']:>10.4f}")

In [ ]:
# 학습 곡선
def plot_training_curves(logs, title, metric_label="accuracy"):
    epochs     = [l["epoch"]           for l in logs]
    train_loss = [l["train"]["loss"]   for l in logs]
    test_loss  = [l["test"]["loss"]    for l in logs]
    train_met  = [l["train"]["metric"] for l in logs]
    test_met   = [l["test"]["metric"]  for l in logs]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    fig.suptitle(title, fontsize=12, fontweight="bold")

    ax1.plot(epochs, train_loss, label="train", color="steelblue", linewidth=2, marker="o")
    ax1.plot(epochs, test_loss,  label="test",  color="#E87B4C",   linewidth=2, marker="o", linestyle="--")
    ax1.set_title("Loss")
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, train_met, label="train", color="steelblue", linewidth=2, marker="o")
    ax2.plot(epochs, test_met,  label="test",  color="#E87B4C",   linewidth=2, marker="o", linestyle="--")
    ax2.set_title(metric_label)
    ax2.set_xlabel("epoch")
    ax2.set_ylabel(metric_label)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_training_curves(logs_mc, "CNN — Multiclass (MNIST 10-class)", metric_label="Accuracy")

## 3. 예측 grid — multiclass

In [ ]:
test_ds = MnistDataset("test", "multiclass", dataset_dir=DATASET_DIR)
N_SHOW  = 16

imgs   = np.stack([test_ds[i][0] for i in range(N_SHOW)])
true_lbls = np.array([test_ds[i][1].argmax() for i in range(N_SHOW)])

result = exp.predictor.predict(imgs)
preds  = result["predictions"]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle("CNN 예측 grid — Multiclass (5 epochs)", fontsize=12, fontweight="bold")

for i, ax in enumerate(axes.ravel()):
    ax.imshow(imgs[i].reshape(28, 28), cmap="gray")
    color = "green" if true_lbls[i] == preds[i] else "red"
    ax.set_title(f"T:{true_lbls[i]} P:{preds[i]}", fontsize=8, color=color)
    ax.axis("off")

plt.tight_layout()
plt.show()

correct = (true_lbls == preds).sum()
print(f"정답 {correct}/{N_SHOW} = {correct/N_SHOW:.0%}")

## 4. CNN 학습 — binary / regression

In [ ]:
# Binary: 홀수(1) vs 짝수(0)
config_bin = {
    "dataset_dir": DATASET_DIR,
    "task":        "binary",
    "model":       "cnn",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  5,
    "lr":          0.01,
}

exp_bin = Experiment(config_bin)
print("CNN binary 학습 시작 (5 epochs)...")
logs_bin = exp_bin.run()

print()
print(f"{'epoch':>5} | {'train loss':>10} {'train acc':>12} | {'test loss':>10} {'test acc':>12}")
print("-" * 58)
for log in logs_bin:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>10.4f} {log['train']['metric']:>12.4f} | "
          f"{log['test']['loss']:>10.4f} {log['test']['metric']:>12.4f}")

In [ ]:
# Regression: label / 9.0
config_reg = {
    "dataset_dir": DATASET_DIR,
    "task":        "regression",
    "model":       "cnn",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  5,
    "lr":          0.001,
}

exp_reg = Experiment(config_reg)
print("CNN regression 학습 시작 (5 epochs)...")
logs_reg = exp_reg.run()

print()
print(f"{'epoch':>5} | {'train loss':>10} {'train R²':>10} | {'test loss':>10} {'test R²':>10}")
print("-" * 55)
for log in logs_reg:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>10.4f} {log['train']['metric']:>10.4f} | "
          f"{log['test']['loss']:>10.4f} {log['test']['metric']:>10.4f}")

In [ ]:
# 3종 task 학습 곡선 비교
all_logs   = [logs_mc,  logs_bin,  logs_reg]
all_titles = ["Multiclass", "Binary", "Regression"]
all_metrics = ["Accuracy", "Binary Accuracy", "R² Score"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("CNN 3종 task 학습 곡선", fontsize=13, fontweight="bold")

for col, (logs, title, met_label) in enumerate(zip(all_logs, all_titles, all_metrics)):
    epochs     = [l["epoch"]           for l in logs]
    train_loss = [l["train"]["loss"]   for l in logs]
    test_loss  = [l["test"]["loss"]    for l in logs]
    train_met  = [l["train"]["metric"] for l in logs]
    test_met   = [l["test"]["metric"]  for l in logs]

    for row, (y_train, y_test, ylabel) in enumerate([
        (train_loss, test_loss,  "loss"),
        (train_met,  test_met,   met_label),
    ]):
        ax = axes[row][col]
        ax.plot(epochs, y_train, label="train", color="steelblue", linewidth=2, marker="o")
        ax.plot(epochs, y_test,  label="test",  color="#E87B4C",   linewidth=2, marker="o", linestyle="--")
        if row == 0:
            ax.set_title(title, fontsize=11)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_xlabel("epoch", fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 예측 grid — 3종 task 비교

In [ ]:
fig, big_axes = plt.subplots(3, 8, figsize=(14, 6))
fig.suptitle("CNN 예측 grid — 3종 task (5 epochs 후)", fontsize=12, fontweight="bold")

task_configs = [
    ("multiclass", exp,     lambda ds, i: ds[i][1].argmax(),          lambda r: r["predictions"]),
    ("binary",     exp_bin, lambda ds, i: int(ds[i][1][0]),           lambda r: r["predictions"].flatten().astype(int)),
    ("regression", exp_reg, lambda ds, i: round(float(ds[i][1][0]) * 9), lambda r: r["predictions"].flatten().astype(int)),
]

for row, (task, experiment, get_true, get_pred) in enumerate(task_configs):
    ds  = MnistDataset("test", task, dataset_dir=DATASET_DIR)
    imgs_row = np.stack([ds[i][0] for i in range(8)])
    trues    = [get_true(ds, i) for i in range(8)]

    result = experiment.predictor.predict(imgs_row)
    preds  = get_pred(result)

    for col in range(8):
        ax = big_axes[row][col]
        ax.imshow(imgs_row[col].reshape(28, 28), cmap="gray")
        color = "green" if trues[col] == preds[col] else "red"
        ax.set_title(f"T:{trues[col]} P:{preds[col]}", fontsize=7, color=color)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(task, fontsize=8)

plt.tight_layout()
plt.show()

## 6. CNN vs MLP — multiclass 비교

In [ ]:
# MLP도 같은 설정으로 학습
config_mlp = {
    "dataset_dir": DATASET_DIR,
    "task":        "multiclass",
    "model":       "mlp",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  5,
    "lr":          0.01,
}

exp_mlp = Experiment(config_mlp)
print("MLP multiclass 학습 시작 (5 epochs)...")
logs_mlp = exp_mlp.run()
print("완료")

In [ ]:
# 파라미터 수 비교
cnn_params = sum(p.size for p in exp.model.params)
mlp_params = sum(p.size for p in exp_mlp.model.params)

print("[ 파라미터 수 비교 ]")
print(f"  CNN 파라미터: {cnn_params:>10,}")
print(f"  MLP 파라미터: {mlp_params:>10,}")
print(f"  CNN / MLP 비율: {cnn_params / mlp_params:.2f}x")
print()
print("[ 최종 test accuracy 비교 ]")
print(f"  CNN: {logs_mc[-1]['test']['metric']:.4f}")
print(f"  MLP: {logs_mlp[-1]['test']['metric']:.4f}")

In [ ]:
# 학습 곡선 비교 시각화
epochs_range = list(range(1, 6))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("CNN vs MLP — Multiclass (5 epochs, lr=0.01, batch=64)",
             fontsize=12, fontweight="bold")

# Loss
ax = axes[0]
ax.plot(epochs_range, [l["train"]["loss"] for l in logs_mc],  color="steelblue",  linewidth=2, marker="o",  label="CNN train")
ax.plot(epochs_range, [l["test"]["loss"]  for l in logs_mc],  color="steelblue",  linewidth=2, marker="o",  linestyle="--", label="CNN test")
ax.plot(epochs_range, [l["train"]["loss"] for l in logs_mlp], color="#E87B4C",    linewidth=2, marker="s",  label="MLP train")
ax.plot(epochs_range, [l["test"]["loss"]  for l in logs_mlp], color="#E87B4C",    linewidth=2, marker="s",  linestyle="--", label="MLP test")
ax.set_title("Cross-Entropy Loss")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(epochs_range, [l["train"]["metric"] for l in logs_mc],  color="steelblue",  linewidth=2, marker="o",  label="CNN train")
ax.plot(epochs_range, [l["test"]["metric"]  for l in logs_mc],  color="steelblue",  linewidth=2, marker="o",  linestyle="--", label="CNN test")
ax.plot(epochs_range, [l["train"]["metric"] for l in logs_mlp], color="#E87B4C",    linewidth=2, marker="s",  label="MLP train")
ax.plot(epochs_range, [l["test"]["metric"]  for l in logs_mlp], color="#E87B4C",    linewidth=2, marker="s",  linestyle="--", label="MLP test")
ax.set_title("Accuracy")
ax.set_xlabel("epoch")
ax.set_ylabel("accuracy")
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. 저장된 outputs 확인

In [ ]:
import os

output_root = "../../outputs"

print("[ outputs/ 폴더 — CNN 실험 결과 ]")
print()
for task in ["multiclass", "binary", "regression"]:
    for model in ["cnn", "mlp"]:
        path = os.path.join(output_root, task, model)
        if os.path.isdir(path):
            files = os.listdir(path)
            total_size = sum(os.path.getsize(os.path.join(path, f)) for f in files)
            print(f"  {task}/{model}/")
            for f in sorted(files):
                sz = os.path.getsize(os.path.join(path, f))
                print(f"    {f}  ({sz/1024:.1f} KB)")
        else:
            print(f"  {task}/{model}/  ← 없음")
    print()

In [ ]:
# 저장된 training_log.png와 predictions.png 표시
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

tasks_to_show = ["multiclass", "binary", "regression"]

for task in tasks_to_show:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"outputs/cnn/{task}/ — 저장된 결과", fontsize=11, fontweight="bold")

    for ax, fname in zip(axes, ["training_log.png", "predictions.png"]):
        fpath = os.path.join(output_root, task, "cnn", fname)
        if os.path.isfile(fpath):
            img = Image.open(fpath)
            ax.imshow(np.asarray(img))
            ax.set_title(fname, fontsize=9)
        else:
            ax.text(0.5, 0.5, "파일 없음", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(fname, fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 8. CuPy fallback 동작 확인

In [ ]:
# CuPy/NumPy fallback: 동일 코드가 두 환경에서 실행됨을 확인
import importlib
import src.models.cnn as cnn_module

print("[ CuPy fallback 확인 ]")
print()
print("cnn.py 상단 fallback 로직:")
print("""
try:
    import cupy as _xp
except ImportError:
    import numpy as _xp
""")

backend_used = cnn_module._xp.__name__
print(f"현재 _xp: {backend_used}")
print()

# CNN 생성 및 forward 실행
cnn_test = CNN(task="multiclass", seed=0)
cnn_test.eval()
x_sample = np.random.randn(2, 784).astype(np.float32)
logits = cnn_test.forward(x_sample)

print(f"CNN.forward() 입력:  numpy, shape={x_sample.shape}")
print(f"CNN.forward() 출력:  {type(logits).__module__}, shape={logits.shape}")
print()
print(f"외부 인터페이스는 항상 NumPy: {type(logits).__module__ == 'numpy'}")
print()
if backend_used == "cupy":
    print("CuPy 가속 사용 중 — conv/pool 연산이 GPU에서 실행됩니다.")
else:
    print("NumPy fallback 모드 — 모든 연산이 CPU에서 실행됩니다.")
    print("코드 변경 없이 CuPy 설치 후 자동으로 GPU 가속됩니다.")

## 9. 정리

**CNN Experiment 설정**
```python
config = {
    "model": "cnn",   # ← 이 한 줄로 CNN 파이프라인 전환
    "task": "multiclass",
    ...
}
exp = Experiment(config)
logs = exp.run()
```

**CNN vs MLP 비교**

| 항목 | CNN | MLP |
|---|---|---|
| 입력 처리 | (N,784) → reshape (N,1,28,28) | (N,784) 그대로 |
| 공간 특징 추출 | Conv2d + MaxPool2d | 없음 |
| 실행 장치 | Conv/Pool: CuPy(GPU), FC: NumPy | NumPy(CPU) |
| 외부 인터페이스 | NumPy 입출력 (MLP와 동일) | NumPy 입출력 |
| Trainer/Evaluator 수정 | 불필요 | — |

**CuPy fallback**
- CuPy 미설치 환경: `_xp = numpy` → 동일 코드 CPU 실행
- CuPy 설치 환경: `_xp = cupy` → Conv/Pool 자동 GPU 가속
- DataLoader, 손실 함수, 메트릭은 수정 불필요

**3종 task 요약**

| task | loss | metric | prediction_mode |
|---|---|---|---|
| multiclass | cross_entropy | accuracy | argmax |
| binary | binary_cross_entropy | binary_accuracy | threshold(0.5) |
| regression | mse | r2_score | round_clip(×9) |